In [1]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total characters: ",len(raw_text))

Total characters:  20479


In [2]:
#creating a basic tokenizer
import re
preprocessed = re.split(r'([,.?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print("Total tokens: ",len(preprocessed)) #without whitespaces

Total tokens:  4649


In [3]:
print("First 10 tokens: ",preprocessed[:10])

First 10 tokens:  ['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius']


In [4]:
#converting tokens into token IDs
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab_size = len(all_tokens)
print("Vocab size: ",vocab_size)

Vocab size:  1161


In [5]:
#creating a vocabulary
vocab = {token: idx for idx, token in enumerate(all_tokens)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i>50:
        break
    

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Carlo;', 25)
('Chicago', 26)
('Claude', 27)
('Come', 28)
('Croft', 29)
('Destroyed', 30)
('Devonshire', 31)
('Don', 32)
('Dubarry', 33)
('Emperors', 34)
('Florence', 35)
('For', 36)
('Gallery', 37)
('Gideon', 38)
('Gisburn', 39)
('Gisburns', 40)
('Grafton', 41)
('Greek', 42)
('Grindle', 43)
('Grindle:', 44)
('Grindles', 45)
('HAD', 46)
('Had', 47)
('Hang', 48)
('Has', 49)
('He', 50)
('Her', 51)


In [6]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1156)
('your', 1157)
('yourself', 1158)
('<|endoftext|>', 1159)
('<|unk|>', 1160)


In [7]:
#tokenizer version 1 - encoder-decoder only
class SimpleTokenizerv1:
    def __init__(self, vocab):
        self.vocab = vocab
        self.inv_vocab = {i: t for t, i in vocab.items()}

    def encode(self, text):
        tokens = re.split(r'([,.?_!"()\']|--|\s)', text)
        tokens = [item.strip() for item in tokens if item.strip()]
        token_ids = [self.vocab[token] for token in tokens]
        return token_ids

    def decode(self, token_ids):
        tokens = [self.inv_vocab[token_id] for token_id in token_ids]
        text = ' '.join(tokens)
        return text

In [28]:
text = "It had always been his fate to have women say such things of him: the fact should be set down in extenuation."

In [27]:
tokens = re.split(r'([,.?_!"()\']|--|\s)', text)
tokens = [item.strip() for item in tokens if item.strip()]
print("Tokens: ",tokens)

Tokens:  ['It', 'had', 'always', 'been', 'his', 'fate', 'to', 'have', 'women', 'say', 'such', 'things', 'of', 'him:', 'the', 'fact', 'should', 'be', 'set', 'down', 'in', 'extenuation', '.']


In [ ]:
tokenizer = SimpleTokenizerv1(vocab)
ids = tokenizer.encode(text)
print("Token IDs: ",ids)

Token IDs:  [58, 522, 154, 214, 559, 439, 1042, 538, 1140, 880, 974, 1022, 738, 556, 1013, 428, 904, 204, 896, 369, 579, 419, 7]


In [9]:
print(tokenizer.decode(ids))

It had always been his fate to have women say such things of him: the fact should be set down in extenuation .


In [10]:
text = "Hello, how are you?"
print(tokenizer.encode(text))

KeyError: 'Hello'

current version of tokenizer can't handle words other than what's there in the vocab

In [23]:
class SimpleTokenizerv2:
    def __init__(self,vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:t for t,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [item if item in self.str_to_int else "<|unk|>" for item in preprocessed]

        token_ids = [self.str_to_int[token] for token in preprocessed]
        return token_ids
    def decode(self, token_ids):
        text = " ".join([self.int_to_str[i] for i in token_ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [14]:
text1 = "Hello, how are you?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))
print(text)

Hello, how are you? <|endoftext|> In the sunlit terraces of the palace.


In [24]:
tokenizer = SimpleTokenizerv2(vocab)
print(tokenizer.encode(text))

[1160, 5, 571, 174, 1155, 10, 1159, 57, 1013, 981, 1009, 738, 1013, 1160, 7]


In [25]:
print(tokenizer.decode([1160, 5, 571, 174, 1155, 10, 1159, 57, 1013, 981, 1009, 738, 1013, 1160, 7]))

<|unk|>, how are you? <|endoftext|> In the sunlit terraces of the <|unk|>.


### BPE Tokenizer